# 01 — What We Built

Before moving on to the library version, we take stock.

Good engineers do not just ship and move on — they understand what they built, where it breaks, and what it would cost to fix those things. This notebook measures the handbuilt system honestly.

## What We Built — Component Inventory

| Component | Where | What it does |
|---|---|---|
| Douglas-Peucker | Module 01 | Reduces point count per line segment |
| LOD pipeline | Module 02 | Produces 4 simplified GeoJSON files |
| Bbox computation | Module 03 | Gets the extent of any feature |
| Bbox intersection | Module 03 | Tests if a feature overlaps the viewport |
| Uniform grid index | Module 04 | Buckets features for fast viewport queries |
| LOD decision function | Module 05 | Selects the right file by zoom level |
| Live map viewer | Module 06 | Wires everything into an interactive display |

Each component was built from scratch. We understand every line.

## Measuring the System

In [2]:
import json
import time
from pathlib import Path

lod_dir  = Path("../../data/lod")
raw_path = Path("../../data/ne_10m_railroads.geojson")

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print(f"{'File':<28} {'Size (MB)':>10} {'Features':>10} {'Total pts':>12} {'Load (s)':>10}")
print("-" * 75)

for label, filename in [("original", None)] + list(lod_files.items()):
    path = raw_path if filename is None else lod_dir / filename
    t0 = time.perf_counter()
    with open(path) as f:
        data = json.load(f)
    load_time = time.perf_counter() - t0
    feats = data["features"]
    n_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    size  = path.stat().st_size / 1_000_000
    print(f"{label:<28} {size:>10.2f} {len(feats):>10,} {n_pts:>12,} {load_time:>10.3f}")

File                          Size (MB)   Features    Total pts   Load (s)
---------------------------------------------------------------------------
original                          39.60     25,413    1,396,480      1.707


FileNotFoundError: [Errno 2] No such file or directory: '../../data/lod/railroads_coarse.geojson'

## Where the System Still Hurts

The viewer works. But it has real limitations. Let's name them honestly.

### Pain Point 1 — Startup Cost

Every session, we load 4 files and build 4 grid indexes. This takes several seconds before the map is usable.

A tile server has no startup cost — tiles are pre-built and stored. The server just reads a file from a database and sends it.

In [ ]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

class GridIndex:
    CELL_SIZE = 10.0
    def __init__(self): self.cells = {}
    def _cells(self, bbox):
        lo, la, hi, ha = bbox; cs = self.CELL_SIZE
        return [(c, r) for c in range(int((lo+180)/cs), int((hi+180)/cs)+1)
                       for r in range(int((la+ 90)/cs), int((ha+ 90)/cs)+1)]
    def build(self, features):
        self.cells = {}
        for i, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)): self.cells.setdefault(cell,[]).append((i,f))

total_startup = 0
for filename in lod_files.values():
    t0 = time.perf_counter()
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    idx = GridIndex()
    idx.build(feats)
    elapsed = time.perf_counter() - t0
    total_startup += elapsed

print(f"Total startup time (load + index build): {total_startup:.2f}s")

### Pain Point 2 — GeoJSON Is Verbose

GeoJSON is human-readable text. Every coordinate is stored as a decimal number string. A production mapping pipeline uses binary encoding (Mapbox Vector Tiles, MVT) which stores coordinates as integers relative to the tile origin — 5–10× smaller than equivalent GeoJSON and much faster to parse.

In [ ]:
# Rough estimate: how large would our files be in a binary format?
# MVT stores coordinates as 2-byte integers per axis
# GeoJSON stores them as ~8-character float strings

GEOJSON_BYTES_PER_COORD = 16   # avg chars for [lon, lat] pair including punctuation
MVT_BYTES_PER_COORD     = 4    # 2 bytes each for x, y as zigzag-encoded varint

for filename in lod_files.values():
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    total_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    actual_mb  = (lod_dir / filename).stat().st_size / 1_000_000
    est_mvt_mb = total_pts * MVT_BYTES_PER_COORD / 1_000_000
    print(f"{filename:<38} actual: {actual_mb:.2f}MB   est MVT: {est_mvt_mb:.2f}MB")

### Pain Point 3 — The Whole File Is Always Resident

To query the fine LOD for Paris, we load the entire `railroads_fine.geojson` into memory — including Australia, South America, and Russia. A tile system would read only the Paris tile from a database, never touching the rest.

Our grid index helps at query time, but the full file still had to load first.

### Pain Point 4 — No Partial Load or Streaming

When the user pans to a new region, we re-query the index immediately — but the data was all loaded at startup. A tile server streams only the tiles the user actually views. If the user never visits Australia, those tiles are never fetched.

## The Decision Inventory

Every system embeds design decisions. Here are ours, stated explicitly:

| Decision | What we chose | What we gave up |
|---|---|---|
| File format | GeoJSON (text) | Binary efficiency |
| Simplification algorithm | Douglas-Peucker via Shapely | Topology-preserving alternatives |
| LOD levels | 4 fixed levels | Continuous zoom-adaptive detail |
| Coarse filter | scalerank ≤ 4 | Coverage in scalerank 5+ regions |
| Spatial index | Uniform 10° grid | Adaptive indexes (R-tree, quadtree) |
| Culling granularity | Feature bbox | True geometry intersection |
| Transition policy | Fixed zoom thresholds | Hysteresis (implemented but not used in final viewer) |
| Memory model | All data loaded at startup | Lazy / tile-based loading |

None of these decisions are wrong. They are appropriate for a teaching system built from scratch. A production system makes different choices for different reasons.

## Exercise A

Measure the peak memory usage of the viewer at startup after all 4 LOD files are loaded and all 4 indexes are built.

How does this compare to just reading the four files without building indexes?

In [ ]:
# Measure peak memory: (a) loading files only, (b) loading + building indexes
import gc
import tracemalloc

def load_all_lod_features():
    # Load all four LOD GeoJSON files and keep their features in memory.
    features_by_lod = {}
    for label, filename in lod_files.items():
        with open(lod_dir / filename) as f:
            features_by_lod[label] = json.load(f)["features"]
    return features_by_lod

# Case A: only load the four GeoJSON files
gc.collect()
tracemalloc.start()
features_only = load_all_lod_features()
current_load_only, peak_load_only = tracemalloc.get_traced_memory()
tracemalloc.stop()

total_features = sum(len(features) for features in features_only.values())
del features_only

gc.collect()

# Case B: load the four GeoJSON files and build one grid index per LOD file
tracemalloc.start()
features_by_lod = load_all_lod_features()
indexes_by_lod = {}
for label, features in features_by_lod.items():
    idx = GridIndex()
    idx.build(features)
    indexes_by_lod[label] = idx
current_with_indexes, peak_with_indexes = tracemalloc.get_traced_memory()
tracemalloc.stop()

index_references = sum(
    len(items)
    for idx in indexes_by_lod.values()
    for items in idx.cells.values()
)

increase_mb = (peak_with_indexes - peak_load_only) / 1_000_000
ratio = peak_with_indexes / peak_load_only if peak_load_only else float("inf")

print(f"Peak memory, loading files only:         {peak_load_only / 1_000_000:.1f} MB")
print(f"Peak memory, loading + grid indexes:    {peak_with_indexes / 1_000_000:.1f} MB")
print(f"Extra memory used by indexes:           {increase_mb:.1f} MB")
print(f"Load + index memory / load-only memory: {ratio:.2f}x")
print(f"Total loaded features:                  {total_features:,}")
print(f"Total grid-index feature references:    {index_references:,}")

print(
"Interpretation:")
print(
    "Reading the files only measures the cost of storing the GeoJSON features in memory. "
    "Building the grid indexes increases memory because each feature is also referenced inside "
    "one or more grid cells. The index makes viewport queries faster, but it does not remove "
    "the cost of loading the full files at startup."
)

## Exercise B

The Railroad LOD system is a handbuilt map-optimization pipeline for displaying railroad data without drawing the full original dataset at every zoom level. It solves the problem of making a large line dataset easier to load, query, and view interactively inside a notebook. The first major component is Douglas-Peucker simplification, which reduces the number of points in each railroad line while preserving its general shape. The second major component is the LOD pipeline, which writes four GeoJSON files at different detail levels: coarse, medium, fine, and extra fine. The third major component is viewport culling, which uses bounding boxes to test whether a feature overlaps the visible map area. The fourth major component is the uniform grid index, which places features into geographic grid cells so the viewer does not have to scan every feature during every pan or zoom. The LOD decision function connects these pieces by choosing the correct detail level based on zoom, and the live viewer uses that choice to display the selected features. The main performance tradeoff is that the system becomes faster during map interaction, but it still pays a startup cost because all LOD files are loaded and indexed first. Another tradeoff is that GeoJSON is easy to inspect and debug, but it is much larger and slower to parse than a binary tile format. If this system had to serve 10 million users, I would replace the all-in-memory GeoJSON workflow with prebuilt vector tiles, server-side tile storage, and lazy loading so each user receives only the tiles needed for the current viewport.

## Check Your Understanding

For a user on a slow connection, I would rank the pain points this way: **1) no partial load or streaming, 2) whole-file loading, 3) verbose GeoJSON format, and 4) startup cost**. No streaming is the biggest problem because the user has to wait for data they may never view, instead of receiving only the visible map area. Whole-file loading is closely related because it forces unnecessary data into memory, while verbose GeoJSON makes the download larger; startup cost matters too, but it is mostly the result of these earlier design choices rather than the root problem.